# Reference Material BAM-N010 
## Hydrodynamic Equivalent Sphere Diameter of Polypropylene Nanoparticles
### Data evaluation for characterization, homogeneity, and stability study

<table align="left" style="text-align: left;">
    <tr><td rowspan="4">
        <img align="left" style="padding: 0em 1em 0 0;" src="info/Bottle_PP-Nanoplastics_IMG_3417.jpg" height="100px" width="100px">
    </td><th>Author:</th><th>Andreas F. Thünemann</th></tr>
    <tr><td>Email: </td><td>andreas.thuenemamm@bam.de</td></tr>
    <tr><td>Phone: </td><td>+49 30 8104 1610 </td></tr>
    <tr><td>Address:</td><td>Unter den Eichen 87, 12205 Berlin</td></tr>
</table>

__The material was discussed at the 111. meeting of the BAM certification committee (ZEBRA) on 17. September 2025__

Subsequently, the data evaluation was revised to meet the committee members' requirements.

The reference material BAM-N010 consists of an aqueous suspension containing polypropylene nanoparticles.  
The material is provided in glass bottles containing 10 mL.  
The goal is to provide the hydrodynamic diameter $D_h$ as a reference.    
Information is additionally given on the polydispersity index PDI and the zeta-potential

## Definitions and Imports

### Loading and configuring required modules

In [ ]:
import os, scipy, glob, sys, re
import datetime
import pandas as pd
import numpy as np
import scipy
from scipy import stats
# The one-way ANOVA tests the null hypothesis that two or more groups have the same population mean. 
# The test is applied to samples from two or more groups, possibly with differing sizes.
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html
from scipy.stats import f_oneway

from lmfit import Minimizer, Parameters
from lmfit.printfuncs import report_fit

import warnings
# Suppress specific warning.
warnings.filterwarnings("ignore", category=SyntaxWarning)

from pathlib import Path
parentDir = str(Path().resolve().parent)
if parentDir not in sys.path:
    sys.path.append(parentDir)

# plotting
import matplotlib
import matplotlib.pyplot as plt
from pyNanoRMcert_tools.plotstyle import configMatplotlibSmall
from pyNanoRMcert_tools.plotstyle import BAMColors
from pyNanoRMcert_tools.plotstyle import BAMMarkerStyles
# rounding
from pyNanoRMcert_tools.round_sig import f_round_sig
from pyNanoRMcert_tools.round_sig import f_round_sig_array

# change font size for axes
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['font.size']= 12

from pyNanoRMcert_tools.utils import store_results, prep_outdir

# Information about software versions
print('Software versions used in this notebook\nPython:', sys.version)
print('scipy: ', scipy.__version__ )
print('numpy: ', np.__version__ )
print('pandas:', pd.__version__ )

## Functions

In [ ]:
def f_values_in_percent(df=None):
    """ Calculated the standard deviation in percent of the mean value. """
    
    df=df.copy()
    list_fractions=[ df[key]['std']/df[key]['mean']*100 for key in df.keys()]
    df.loc[3]=list_fractions
    df.rename(index={3:'std%'}, inplace=True)
    return df

In [ ]:
def f_read_DLS(data_dir, files='p*.xlsx', verbose=True):
    ''' Read data from DLS measurements measured with the Anton Paar Litesizer 500 instrument 
    *data_dir* is the directory containing the data files'''

    file_list=glob.glob(os.path.join(data_dir, files))
    print('len(file_list) =', len(file_list),'(number of files)\n')
    
    # dictionary with all data
    d_data={}
    for item in file_list:
        if verbose:
            print (item)
        
        match1 = re.search(r'P\d+', item)
        if match1: 
            match=match1

        match2 = re.search(r'Z\d+', item)
        if match2:
            match=match2
                
        if match:
            file_ID=match.group()
            df=pd.read_excel(item)
            # Generate alphabetical column names
            alphabetical_columns = [chr(i) for i in range(ord('A'), ord('A') + len(df.columns))]
            # Rename the columns
            df.columns = alphabetical_columns
            df.replace('nan', np.nan, inplace=True) # Replace string 'nan' with actual np.nan
            #df.fillna(0, inplace=True) # Fill all NaN values with a desired value, e.g., 0
            
            # numerische Spalten
            num_cols = df.select_dtypes(include=["number"]).columns
            df[num_cols] = df[num_cols].fillna(0)
            
            # String-Spalten
            str_cols = df.select_dtypes(include=["string"]).columns
            df[str_cols] = df[str_cols].fillna("")
            
    
            d_data[file_ID]=df
        else:
            print('something went wrong:', item)
    return d_data

#d_data1=f_read_DLS(basepath / '2026-05-07_BAM-N010', files='[pP]*.xlsx', verbose=False)
#d_data1

In [ ]:
def f_DLS_results(d_data):
    
    ''' Read the relevant data from the measurement files. 
    *d_data* is a dictionary containing the data files
    '''
    
    df_r=pd.DataFrame()
    
    l_measurement_name, l_date, l_sample_ID, l_comment, l_Dh, l_PDI, l_temperature =  [], [], [], [], [], [], []
    
    for item in d_data.keys():
        
        l_measurement_name.append(d_data[item]['B'].iloc[0])
        comment=d_data[item]['B'].iloc[3]
        
        # Replace non-breaking spaces with regular spaces
        comment = comment.replace('\n', ' ')

        #print('comment', comment)
        try:
            sample_ID = re.search(r'ID-Nr\.\s*:\s*(\d+)', comment).group(1)
            l_sample_ID.append(sample_ID)
        except:
            pass
        try:
            sample_ID=re.search(r'ID(\d+)', comment).group(1)
            l_sample_ID.append(sample_ID)
        except:
            pass
        try:
            sample_ID=re.search(r'ID (\d+)', comment).group(1)
            l_sample_ID.append(sample_ID)
        except:
            pass
    
    
        l_comment.append(comment)
    
        l_Dh.append(d_data[item]['C'].iloc[5])
        l_PDI.append(d_data[item]['C'].iloc[6]/100)
    
        # find date of measurement
        date_str=d_data[item][d_data[item]['B'].str.contains("Start time", case=False, na=False)]['C'].values[0]
        date_str=str(date_str)
        try:
            date=datetime.datetime.strptime(date_str, "%m/%d/%Y %I:%M:%S %p")
        except:
            date=datetime.datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S')
            
        l_date.append(date.date())
        temperature=d_data[item][d_data[item]['B'].str.contains("Temperature", case=False, na=False)]['C'].values[0]
        l_temperature.append(temperature)
                        
    df_r['measurement_name']=  l_measurement_name
    #df_r['sample_ID'] = l_sample_ID
    df_r['ID'] = l_sample_ID
    df_r['date of measurement']=    l_date
    df_r['temperature'] = l_temperature
    df_r['Dh']=         l_Dh
    df_r['PDI']=        l_PDI
    df_r['comment']=    l_comment
    df_r
    
    return df_r

#f_DLS_results(d_data1)

In [ ]:
def f_zeta_results(d_data):
    ''' Read the relevant data from the measurement files. 
    *d_data* is a dictionary containing the data files
    '''
    
    df_r=pd.DataFrame()
    
    l_measurement_name, l_date, l_sample_ID, l_comment, l_zeta, l_PDI, l_temperature =  [], [], [], [], [], [], []
    
    for item in d_data.keys():
        df=d_data[item]
        
        y=df[df['A'].str.contains("Measurement name", case=False, na=False)]['B'].values[0]
        l_measurement_name.append(y)
        
        y=df[df['B'].str.contains("Start time", case=False, na=False)]['C'].values[0]        
        try:
            l_date.append(y.date())
        except:
            y=datetime.datetime.strptime(y, '%m/%d/%Y %I:%M:%S %p')
            l_date.append(y.date())
    
        comment=df[df['A'].str.contains("Comment", case=False, na=False)]['B'].values[0]
        l_comment.append(comment)
        try:
            start=re.search(r'\d+', comment).start()
            y=comment[start:start+3]
            l_sample_ID.append(y)
        except:
            print('no ID number found')
        y=df[df['B'].str.contains("Mean zeta potential", case=False, na=False)]['C'].values[0]
        l_zeta.append(y)
        
        y=df[df['B'].str.contains("Target temperature", case=False, na=False)]['C'].values[0]
        l_temperature.append(y)
    
    df_r['measurement_name']=  l_measurement_name
    df_r['ID'] = l_sample_ID
    df_r['date of measurement']=    l_date
    df_r['temperature'] = l_temperature
    df_r['zeta']=         l_zeta
    df_r['comment']=    l_comment

    return df_r

### ANOVA
- Nul hypothesis: The data from day 1, 2 and 3 have the same mean
- Sample data for control of the implementation are from

ISO GUIDE 35:2017(E) Reference materials — Guidance for characterization and assessment of homogeneity and stability  
See page 89: Table C.1 — Measurement data of a between-unit homogeneity study
of chromium in soil

In [ ]:
def f_ISO_Guide_35_ANOVA(df):
    """Perform one-way ANOVA according to ISO Guide 35
    
    The one-way ANOVA tests the null hypothesis that two or more groups have the same population mean. 
    The sample data for control are from
    ISO GUIDE 35:2017(E) Reference materials — Guidance for characterization and assessment of homogeneity and stability
    Page 89, Table C.1 — Measurement data of a between-unit homogeneity study of chromium in soil
    
    see also
    https://en.wikipedia.org/wiki/One-way_analysis_of_variance
    """
    df = df.copy() # a copy of the dataframe provided

    # Step 1: Calculate the mean within each group;  groups are the bottles:
    cols = list(df.keys())
    a = len(df) # number of groups
    n = 3 # where n is the number of data values per group. 
    print('Number of groups a =', a, '\nNumber of data per group n =',n )
    
    # Create a summary table as Excel produces it for "Anova: Single Factor"
    d_sums, d_means = {}, {} # dictionary containing mean of each group
    for i in range(a):
        # sums
        d_sums['Y'+str(i)] =(df.iloc[i][cols[1]]+df.iloc[i][cols[2]]+df.iloc[i][cols[3]])
        # means
        d_means['Y'+str(i)]= (df.iloc[i][cols[1]]+df.iloc[i][cols[2]]+df.iloc[i][cols[3]])/n
        
    # Step 2: Calculate the overall mean: 
    Y = sum([ d_means[key] for key in d_means.keys()])/len(d_means)
    #print('Overall mean =',f_round_sig(Y,5))
    
    # Step 3: Calculate the "between-group" sum of squared differences: 
    SS_between = sum([ n*(d_means[key]-Y)**2 for key in d_means.keys()])
    # The between-group degrees of freedom is one less than the number of groups
    df_between = a-1
    MS_between = SS_between/df_between
    #print('Source of variance between-group: SS =',f_round_sig(SS_between,5), ', df =', df_between, ', MS =',f_round_sig(MS_between,5))

    # Step 4: Calculate the "within-group" sum of squares. 
    # sum of the values within each group
    df['Sum'] = d_sums.values()
    # mean value within each group
    df['Mean'] = d_means.values()
    # sum of squares within each group
    df['SS'] = (df[df.keys()[1]]-df['Mean'])**2+(df[df.keys()[2]]-df['Mean'])**2+(df[df.keys()[3]]-df['Mean'])**2
    # standard deviation of the mean within each group
    df['Variance'] = df['SS']/(n-1)
    df['Sigma'] = np.sqrt(df['Variance'])
    # sum of squares within group
    SS_within = df['SS'].sum()
    # The within-group degrees of freedom is a*(n-1)
    df_within = a*(n-1)
    # Thus the within-group mean square value is 
    MS_within = SS_within/df_within
    #print('Source of variance within-group:  SS =',f_round_sig(SS_within,5), ', df =', df_within, ', MS =',f_round_sig(MS_within,5))
    F_value = MS_between/MS_within
    #print('F_value = ', f_round_sig(F_value,5))

    # find the critical F-Value
    # https://stackoverflow.com/questions/39813470/f-test-with-python-finding-the-critical-value
    import scipy.stats
    # Confidence level of 95% is q=1.-0.05
    F_crit = scipy.stats.f.ppf(q=1-0.05, dfn=df_between, dfd=df_within)
    #print("F_crit  = ", f_round_sig(F_crit,5), "(critical F-Value, corresponding to a confidence level of ",
    #scipy.stats.f.cdf(F_crit, dfn=df_between, dfd=df_within)*100,'%)')

    # check of F value and determination of p-value 
    array4f_oneway = [[df.iloc[i][cols[1]], df.iloc[i][cols[2]],  df.iloc[i][cols[3]]] for i in range(len(df))]
    F_value_2, p_value = f_oneway(*array4f_oneway) 
    #print('p-value =', p_value)
    
    # The between-unit standard deviation (see. formula C.2 at page 90 of ISO GUIDE 35:2017(E))
    if MS_between > MS_within:
        s_bb = np.sqrt((MS_between-MS_within)/n)
    else: s_bb = 0.
    
    # The repeatability standard deviation (see formula C.3 at page 90 of ISO GUIDE 35:2017(E)))
    s_r  = np.sqrt(MS_within)
    
    # Table according to ISO Guide 35:2017(E) Table C.2, page 89
    df_ANOVA_table = pd.DataFrame( 
        {   'Overall Mean':[f_round_sig(Y,5),'',''],
            'Overall Std': [f_round_sig(
                    np.sqrt((SS_between+SS_within)/(df_between+df_within)),3),'',''],
            'Source of variation':['Between bottles','Within bottles','Total'], 
            'Sum of squares':[
                f_round_sig(SS_between,5),
                f_round_sig(SS_within,5),
                f_round_sig((SS_between+SS_within),5)
                ], 
            'Degrees of freedom':[df_between,df_within,df_between+df_within], 
            'Mean square':[ f_round_sig(MS_between,5), f_round_sig(MS_within,5), ''],
            'Standard deviation':[f_round_sig(s_bb,4), f_round_sig(s_r,4),' '],
            'F':[f_round_sig(F_value,5),'',''], 
            'Fcrit':[f_round_sig(F_crit,5),'',''],
            'p-value':[p_value,'',''],
        })
    
    return df, df_ANOVA_table

# Data import

In [ ]:
basepath = Path(r"/mnt/fb65/PAZ/NP-Referenzmaterial/BAM-N010_PP/06_Datenauswertung/data")
resultspath = basepath.parent / 'results_ib'
resultspath.mkdir(parents=True, exist_ok=True)

## 2026-05-07 study after discussion with Kristin Vogel
10 samples were measured, each three times  
Disposal cuvettes were used

In [ ]:
d_data1=f_read_DLS(basepath / '2026-05-07_BAM-N010', files='[pP]*.xlsx', verbose=False)
df_2026_05_07=f_DLS_results(d_data1)
df_2026_05_07.rename(columns={'Dh': 'D', 'sample_ID': 'ID'}, inplace=True)
df_2026_05_07['Device']='Litesizer 500'
display(df_2026_05_07.head(2))
df_2026_05_07[['D','PDI']].describe().head(3)

## 2025-09-25 homogeneity study 
Note: Omega cuvettes are used

In [ ]:
d_data1=f_read_DLS(basepath / 'data_2025-09-25_BAM-N010', files='[pP]*.xlsx', verbose=False)
df_2025_homogeneity=f_DLS_results(d_data1)
df_2025_homogeneity.rename(columns={'Dh': 'D', 'sample_ID': 'ID'}, inplace=True)
df_2025_homogeneity['Device']='Litesizer 500'
display(df_2025_homogeneity.head(2))
df_2025_homogeneity[['D','PDI']].describe().head(3)

## Data from 2025-06-30 and 2025-07-07
Before and after heat treatment for 90 min at 121°C

In [ ]:
d_data1=f_read_DLS(basepath / 'data_2025-06-30', files='[pP]*.xlsx', verbose=False)
df_D1=f_DLS_results(d_data1)
d_data2=f_read_DLS(basepath / 'data_2025-07-07', files='[pP]*.xlsx', verbose=False)
df_D2=f_DLS_results(d_data2)
df_D=pd.concat([df_D1,df_D2])
df_D.rename(columns={'Dh': 'D', 'sample_ID': 'ID'}, inplace=True)
df_2025_heat=df_D.copy()
df_2025_heat['Device']='Litesizer 500'

fig, ax = plt.subplots()
ax.errorbar(df_D[0:10].index, df_D['D'][0:10], **BAMMarkerStyles.blue , label='before heat treatment')
ax.errorbar(df_D[0:10].index, df_D['D'][10:20], **BAMMarkerStyles.red, label='after heat treatment 1')
ax.errorbar(df_D[0:10].index, df_D['D'][20:30], **BAMMarkerStyles.green, label='after heat treatment 2')
ax.set(
    xlabel='Sample',
    ylabel='$D_h$ (nm)',
    ylim=(100,250)
)
ax.legend(frameon=False)

display(df_2025_heat.head(2))

df_D[['D','PDI']].describe().head(3)

In [ ]:
d_data1=f_read_DLS(basepath / 'data_2025-06-30', files='[zZ]*.xlsx', verbose=False)
df_zeta1=f_zeta_results(d_data1)
d_data2=f_read_DLS(basepath / 'data_2025-07-07', files='[zZ]*.xlsx', verbose=False)
df_zeta2=f_zeta_results(d_data2)
df_zeta=pd.concat([df_zeta1,df_zeta2])
df_zeta.reset_index(inplace=True, drop=True)
df_zeta_2025_heat=df_zeta.copy()
df_zeta_2025_heat['Device']='Litesizer 500'
df_zeta_2025_heat.head(2)

## Data - 2023-08-16 - 2024-01-04 - 2025-03-20
- measurement with Litesizer 500

In [ ]:
d_data1=f_read_DLS(basepath / 'data_2023', files='[pP]*.xlsx', verbose=False)
df_2023_24_25 =f_DLS_results(d_data1)
df_2023_24_25.rename(columns={'Dh': 'D'}, inplace=True)
#display(df_2023_24_25)
display(df_2023_24_25[['D','PDI']].describe().head(3))
d_data1=f_read_DLS(basepath / 'data_2023', files='[zZ]*.xlsx', verbose=False)
df_zeta_2023_24_25=f_zeta_results(d_data1)
df_zeta_2023_24_25['Device']='Litesizer 500'
display(df_zeta_2023_24_25.head(2))
display(df_zeta_2023_24_25['zeta'].describe().head(3))

## 2022 Homogeneity study ZetaSizer
- performed with ZetaSizer in January 2022
### Overview of the parameter values on each measurement day
- Production of 3 data frames associated to day 1, day 2 and day 3 of the homogeneity study

In [ ]:
list_days = ['day1','day2','day3']

l_dfs=[]
for item in list_days:
    # read data
    df = pd.read_excel(basepath / 'data_homogeneity_2022' / 'ZetaSizer_NanoPP.xlsx',
                                   sheet_name='PP_homogeneity_'+item, skiprows=1, nrows=20)
    # read the date of measurement
    df_date = pd.read_excel(basepath / 'data_homogeneity_2022' / 'ZetaSizer_NanoPP.xlsx',
                               sheet_name='PP_homogeneity_'+item, skiprows=0, nrows=1, header=None)

    df['Date'] = df_date[1].dt.date[0]
    df['Day']  = "Day "+ item[-1]
    df.rename(columns={'ID': 'Sample ID','diameter (nm)': 'D', 'Zeta Potential (mV)':'zeta'}, inplace=True)

    l_dfs.append(df)
df_2022_homogeneity=pd.concat(l_dfs, ignore_index=True)
df_2022_homogeneity['Device']='ZetaSizer Nano ZS'

print(f"Number of entries: {len(df_2022_homogeneity)}")
display(df_2022_homogeneity.head(3))
display(df_2022_homogeneity[['D','PDI','zeta']].describe().head(3))

df_char_ZetaSizer=df_2022_homogeneity[df_2022_homogeneity['Date']==datetime.date(2022, 1, 24)]
df_char_ZetaSizer.head(2)

## Data from the ALV instrument

In [ ]:
def f_find_values(filename, marker='R_h(90)', verbose=True):
    """ Find results in the DLS results file of the ALS instrument """

    with open(filename) as fd:
        lines = fd.readlines()
    for line in lines:
        if marker in line:
            if verbose:
                print(f'Found: {line}')
            # Find numbers that appear in line
            #match = re.search(r"R_h.*?=\s*([-+]?\d*\.\d+|\d+)\s*\+/-\s*([-+]?\d*\.\d+|\d+)", line)

            #convert 'marker' to regular expression
            s=marker.replace("(", r"\(")
            s=s.replace(")", r"\)")
            match = re.search(s+r".*?=\s*([-+]?\d*\.\d+|\d+)\s*\+/-\s*([-+]?\d*\.\d+|\d+)", line)
            
            if match:
                value = float(match.group(1))
                uncertainty = float(match.group(2))
                if verbose:
                    print(f'{marker} = {value} +/- {uncertainty}')
            else:
                print(f"No numbers found after {marker}")
            
    return marker, value, uncertainty

In [ ]:
def f_find_measurement_ID(filename, marker='Measurement ID:', verbose=True):
    """ Find DLS measurement ID of the ALS instrument """

    print('filename', filename)

    with open(filename) as fd:
        lines = fd.readlines()
    for line in lines:
        if marker in line:
            if verbose:
                print(f'Found: {line}')
            match = re.search(r'\d+', line)
            
            if match:
                ID = int(match.group())
            else:
                print(f"No ID found after {marker}")       
    return ID

#f_find_measurement_ID(file_path, marker='Measurement ID', verbose=True)

In [ ]:
#directory=os.path.join(path, directories[1])

def f_search_ID(directory, marker= 'NanoPE ID', verbose=False):
    '''search for sample ID'''
    
    # Loop through the files in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.ASC'):
            if verbose:
                print("First .asc file found:", os.path.join(directory, filename))
            with open(os.path.join(directory, filename), encoding="cp1252") as fd:
                lines = fd.readlines()
                for line in lines:
                    #print('line = ', line)
                    if marker in line:
                        #print(line)
                        match=match = re.search(r'ID\d+', line)
                        #print('match =', match)
                        if match:
                            result = match.group(0)
                            #print('result =', result)
                        else:
                            print("Pattern not found.")
                break
    return result

file_name='106'
f_search_ID(basepath / 'data_2025-07-01_ALV' / file_name, marker= 'NanoPP ID', verbose=False)

In [ ]:
d_ALV={}
for date_data in ['data_2025-07-01_ALV']:
    #path=os.path.join('data', 'ALV', '2025 04 30')
    path=basepath / date_data
    # Get all directories that contain measurements
    directories = [name for name in os.listdir(path) if os.path.isdir(os.path.join(path, name))]
    #display(directories)
    
    l_df=[]
    for i in range(0, len(directories)):
        file_name=directories[i]+'.txt'
        file_path=os.path.join(path, directories[i], file_name)
    
        print('file_path', file_path)
        
        d_results={}
        item=f_search_ID(os.path.join(path, directories[i]), marker= 'NanoPP ID', verbose=False)
        d_results['ID']= [item]
        
        item=f_find_measurement_ID(file_path, marker='Measurement ID', verbose=False)
        d_results['measurement_ID']=item
        
        for item in ['R_h(90)', 'PDI(90)', 'R_h(173)', 'PDI(173)']:
            result = f_find_values(file_path,  marker=item, verbose=False)
            d_results[result[0]] = [result[1]]
            if item in ['R_h(90)', 'R_h(173)']:
                d_results['u'+result[0]] = [result[2]]
            df_results=pd.DataFrame(d_results)
        l_df.append(df_results) 
    
    df_ALV=pd.concat(l_df, ignore_index=True)
    df_ALV['D']=2*df_ALV['R_h(173)']
    df_ALV['uD']=2*df_ALV['uR_h(173)']
    
    df_ALV['PDI']=2*df_ALV['PDI(90)']
    df_ALV['ID']=df_ALV['ID'].str[2:].astype(int)
    df_ALV=df_ALV.sort_values(by='ID')
    df_ALV.reset_index(inplace=True, drop=True)
    #display(df_ALV)
    d_ALV[date_data]=df_ALV


In [ ]:
df_2025_ALV=d_ALV['data_2025-07-01_ALV']
df_2025_ALV['Device']='ALV 7004'
display(df_2025_ALV)
display(df_2025_ALV[['D','PDI']].describe().head(3))

# Characterization
## Standard uncertainty due to characterization $u_{char}$

The hydrodynamic diameter of BAM-N010 was determined with dynamic light scattering in aqueous dispersions with three different devices.
1. ZetaSizer Nano ZS (Malvern Panalytical, follows ISO standard 22412:2017)
2. Litesizer 500 (Anton Paar AG, follows ISO standard 22412:2017)
3. ALV 7004 (ALV Langen)

The measurements of the dispersions were carried out directly after opening the vials with a volume of 1 ml without further treatment at a temperature of 25°C. Ten randomly selected bottles were measured three times each.

The assigned value $y_{char}$ for the $D_h$, PDI, and zeta potential was calculated from the average values $y_i$ obtained from the
measurements of each device and the number of devices $p = 3$ as 
\begin{equation}
y_{char}=\frac{\sum y_i}{p}.
\end{equation}

The $u_{\textrm{char}}$ is then calculated according to equation (A.4) in ISO Standard as
\begin{equation}
u_{char}= \frac{1}{\sqrt{p}} \sqrt{ \frac{ \sum{ (y_i-y_{char})^2} }{p-1}}.
\end{equation}

-  Kristin Vogel wants to see the single measurements, according to her comments in the report

In [ ]:
df_char_litesizer = df_2025_homogeneity[0:10][['ID','D']]#.sort_values(by="ID")
df_char_litesizer["D"] = df_char_litesizer["D"].round(1)
df_char_litesizer.insert(0, "Device 1", "Litesizer 500")
df_char_litesizer.rename(columns={"ID": "ID 1", "D": "Dh 1"}, inplace=True)

df_char_alv = df_2025_ALV[['ID','D']]
df_char_alv["D"] = df_char_alv["D"].round(1)
df_char_alv.insert(0, "Device 2", "ALV 7004")
df_char_alv.rename(columns={"ID": "ID 2", "D": "Dh 2"}, inplace=True)

df_char_zeta = df_2022_homogeneity[0:10][['Sample ID','D']]
df_char_zeta["D"] = df_char_zeta["D"].round(1)
df_char_zeta.insert(0, "Device 3", "ZetaSizer Nano ZS")
df_char_zeta.rename(columns={"Sample ID": "ID 3", "D": "Dh 3"}, inplace=True)

df_char = pd.concat((df_char_litesizer, df_char_alv, df_char_zeta), axis=1)
df_char

### Export as word table

In [ ]:
# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

def df_to_word(df: pd.DataFrame, outpath: Path, save_file=True,
               include_index=False, description=""):
    doc = Document()
    title=doc.add_heading(level=1)
    run= title.add_run(description)
    title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

    run.font.name='BAM Klavika Light'
    run.font.size = Pt(10) # Optional: adjust size
    run.font.color.rgb = RGBColor(0,0,0) # Black

    columns = tuple(df.columns)
    if include_index:
        columns = (df.index.name,) + columns

    table = doc.add_table(rows=1, cols=len(columns))
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    for i, col_name in enumerate(columns):
        paragraph = hdr_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        run = paragraph.add_run(col_name)
        run.font.name = 'BAM Klavika Light'#'Arial'
        run.font.size = Pt(10)  # Optional: set font size

    #for col in df.select_dtypes(include='float'):

    #for col in df[['before heat','after heat 1', 'after heat 2','Sum','Mean','SS']].select_dtypes(include='float'):
    #    df[col] = df[col].map(lambda x: f"{x:03.1f}")

    #df['Variance'] = df['Variance'].map(lambda x: f"{x:02.1f}")
    #df['Sigma'] = df['Sigma'].map(lambda x: f"{x:01.1f}")

    for idx, row in df.iterrows():
        row_cells = table.add_row().cells
        tbl_row = tuple(row)
        if include_index:
            tbl_row = (idx,) + tbl_row
        for i, value in enumerate(tbl_row):
            paragraph = row_cells[i].paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
            run=paragraph.add_run(str(value))
            run.font.name = 'BAM Klavika Light'#'Arial'
            run.font.size = Pt(10)  # Optional: set font size

    if not save_file:
        return
    doc.save(outpath)
    print(f'Table saved as {outpath}')

df_to_word(df_char, resultspath / "characterization_table.docx",
           description="Characterization of the hydrodynamic diameters of BAM-N010 with three different DLS devices. Provided are the devices, the sample IDs, and the corresponding Dh values.")

In [ ]:
df_char_mean=df_char[['Dh 1', 'Dh 2', 'Dh 3']].describe().iloc[1:3].round(1)
df_char_mean.columns = (df_char['Device 1'][0], df_char['Device 2'][0], df_char['Device 3'][0])
df_char_mean.index = ['Dchar, mean', 'Standard deviation']
df_char_mean.index.names = ('Device',)
df_char_mean

In [ ]:
df_to_word(df_char_mean, resultspath / "characterization_table_mean.docx",
           include_index=True,
           description="Mean values of the hydrodynamic equivalent spherical diameter and standard deviation for measurements from three different instruments.")

In [ ]:
def f_mean_std(l_values, par, digits=3):
    ''' ISO Guide 35:2017
    A.2.5 Assigned uncertainty 
    A.2.5.3 Evaluation without the laboratories' uncertainty
    Note: This is smaller than the standard deviation of the mean values
    '''
    l_values=np.array(l_values)
    p=len(l_values)
    y=l_values.sum()/p
    
    l_sum=[]
    for item in l_values:
        l_sum.append(  (item-y)**2)
    
    # standard deviation of the p data set mean values
    s=np.sqrt(np.sum(l_sum)/(p-1))
    u_char=s/np.sqrt(p)
    u_rel=np.abs(u_char/y*100)

    df_res=pd.DataFrame()
    df_res['parameter']=[par]
    df_res['y_char']= [round(y,digits)]
    
    df_res['u_char']= [round(u_char,digits)] 
    df_res['u_rel']= [round(u_rel,digits)]

    return df_res


l_values=list(df_char_mean.loc["Dchar, mean"][:2].values)
display(l_values)

l_dfs=[]
d_ychar={}
d_ychar['D']=f_mean_std(l_values, 'D', digits=1)
display(d_ychar['D'])

y_char=d_ychar['D']['y_char'][0]
u_char=d_ychar['D']['u_char'][0]
print(f'y_char = {y_char}')
print(f'u_char = {u_char}')

# 2025 Homogeneity study 
- Litesizer 500

In [ ]:
df_2025_homogeneity.head(2)

In [ ]:
def f_prepare_for_ANOVA(df=None, par=None):
    """Prepare the data for ANOVA evaluation
    
    example C.1 Homogeneity study; ISO 33405_2024(en)
    """
    
    df = df.copy()
    
    df1 = df.loc[0:9].copy()
    df1.sort_values(by='ID', inplace=True)
    df1.reset_index(inplace=True)
    df_4ANOVA = df1[['ID',par]].copy()
    df_4ANOVA.rename(columns={'ID': 'Sample ID', par:'Result 1'}, inplace=True)

    df2 = df.loc[10:19].copy()
    df2.sort_values(by='ID', inplace=True)
    df2.reset_index(inplace=True)
    df_4ANOVA['Result 2'] = df2[par]

    df3 = df.loc[20:29].copy()
    df3.sort_values(by='ID', inplace=True)
    df3.reset_index(inplace=True)
    df_4ANOVA['Result 3'] = df3[par]
    
    return df_4ANOVA

df_ISO_33405=f_prepare_for_ANOVA(df_2025_homogeneity,par='D')

display(df_ISO_33405)
df, df_ANOVA_table = f_ISO_Guide_35_ANOVA(df_ISO_33405)
df_homogeneity_data=df.copy()
print("\nANOVA according to standard ISO 33405:2024(en), example Page 84, Table C.1")
print('\nHydrodynamic diameter')
df_ANOVA_table

## Export as Word file

In [ ]:
df_to_word(df_homogeneity_data.round(1), resultspath / "homogeneity_data.docx",
           include_index=False,
           description="Measurement data of the hydrodynamic diameter of the between-unit homogeneity study. Units of Result 1, Result 2, Result 3, Sum, Mean, and Sigma are in nm. Units of SS and Variance are nm2")

### Standard uncertainty due to homogeneity study $u_{\textrm{hom}}$

\begin{equation}
u_{\textrm{hom}}= \sqrt{u_{bb}^2+ u_{wb}^2}
\end{equation}

The standard uncertainty due to between-bottle variation is
\begin{align}
u_{bb}&=s_{bb}\\
s_{bb}&=\sqrt{\max ({  \frac{M_{between}-M_{within}}{n} },0 ) }
\end{align}

The standard deviation of repeatability within bottles is
\begin{align}
u_{wb}&=s_r\\ 
s_r&=\sqrt{M_{within}}
\end{align}

In [ ]:
def f_calc_uncertainty(df_ANOVA):
    """ Calculation of the uncertainty of a homogeneity study

    """
    # Overall Mean
    y_hom= df_ANOVA['Overall Mean'][0]
    print(f'y_hom = {y_hom}')
    
    # standard deviation within bottles
    s_r=df_ANOVA['Standard deviation'][1]
    u_wb=s_r
    print(f'u_wb = {u_wb:2.3f}')
    
    n=3  # number of days
    N=20 # number of bottles
    M_between=df_ANOVA['Mean square'][0]
    M_within=df_ANOVA['Mean square'][1]
    
    y_brackets=(M_between-M_within)/n
    s_bb= np.sqrt( np.max([y_brackets, 0.]))
    u_bb=s_bb
    print(f'u_bb = {u_bb:2.3f}')
    
    u_hom = np.sqrt(u_wb**2 + u_bb**2)
    print(f'u_hom = {u_hom:2.3f}\n')

    return y_hom, u_hom

y_hom, u_hom = f_calc_uncertainty(df_ANOVA_table)

In [ ]:
def f_plot_homogeneity(df, df_ANOVA=None, par='D', save_ANOVA=False):
    """ Plot the results of the homogeneity study. """
    
    fig, ax = plt.subplots(1,1, figsize=[6.4, 4.8])
    
    # --- Diameter ---
    ax.errorbar(list(df.index), 'Mean', 'Sigma', data=df, **BAMMarkerStyles.black)#, label='data')
    
    # one sigma
    x_min = 0; x_max = 10
    x = np.linspace(x_min, x_max,50)
    y_mean = np.ones(len(x))* df_ANOVA['Overall Mean'][0]
    ax.errorbar(x, y_mean, ls='-.', lw=2, color=BAMColors.red)

    y_min = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]-df_ANOVA['Overall Std'][0])
    y_max = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]+df_ANOVA['Overall Std'][0])
    #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.red)
    #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.red)
    
    # two sigma
    y_min = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]-2*df_ANOVA['Overall Std'][0])
    y_max = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]+2*df_ANOVA['Overall Std'][0])
    ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.blue)
    ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.blue)
    
    # mean value +- sigma
    x = [22.]; y = df_ANOVA['Overall Mean'][0]
    uy = df_ANOVA['Overall Std'][0]
    #ax.errorbar(x, y, uy, **marker_style_red)#, label=r'$\left<D\right>\pm 1\sigma$')
    
    # mean value +- 2 sigma
    x = [24.]; 
    y = df_ANOVA['Overall Mean'][0]
    uy = 2*df_ANOVA['Overall Std'][0]
    #ax.errorbar(x, y, uy, **marker_style_blue)#, label=r'$\left<D\right>\pm 2\sigma$')
    #ax.set_xlabel('Sample ID', fontsize=15)
    
    #ax.set_xticks(list(df.index) + [22.,24.])
    ax.set_xticks(list(df.index))
    
    #ax.legend(loc='upper right')
    #ax.set_ylim(y-1.5*uy,y+1.5*uy)
    
    # parameter-dependent axis labelling
    if par=='D':
        #list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        #list_xticks=list(df['Sample ID'].values) + ['',r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        list_xticks=list(df['Sample ID'].values)
        ax.set_ylabel(r'$D_\text{h}$ (nm)', fontsize=15)
        #ax.set_ylim(140, 240)
    
    if par=='PDI':
        #list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        #list_xticks=list(df['Sample ID'].values) + [r'',r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        list_xticks=list(df['Sample ID'].values) 
        ax.set_ylabel(r'{}'.format(par), fontsize=15)
        ax.set_ylim(0,.2)
        
    if par=='zeta':
        #list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        #list_xticks=list(df['Sample ID'].values) + [r''.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        list_xticks=list(df['Sample ID'].values)
        ax.set_ylabel(r'Zeta potential (mV)', fontsize=15)
        #ax.set_ylim(-60,-30)

    ax.set_xticklabels(list_xticks, rotation = 60)
    ax.set_xlabel("Sample ID")
        
    if save_ANOVA:
        store_results(resultspath, f"ANOVA_{par}",
                      (df, df_ANOVA), ("Data", "Table"), saveplot=True)
    plt.show()
    return

f_plot_homogeneity(df=df_homogeneity_data, df_ANOVA=df_ANOVA_table, par='D', save_ANOVA=True)
print("=> What are the .xlsx files used for?")

# Long-term stability study

In [ ]:
# Prepare the Dataframes to the same format before concatenation.

df_2026_05_07.rename(columns={'date of measurement':'Date', 'ID': 'Sample ID'}, inplace=True)

df_2023_24_25.rename(columns={'date of measurement':'Date', 'ID': 'Sample ID'}, inplace=True)

df_2025_ALV.rename(columns={'ID':'Sample ID'},inplace=True)
df_2025_ALV['Date']=datetime.date(2025,7,1)

df_2025_homogeneity.rename(columns={'date of measurement':'Date', 'ID': 'Sample ID'}, inplace=True)

df_long_term=pd.concat([
df_2022_homogeneity[['Sample ID', 'D','PDI','Date']],
df_2023_24_25[['Sample ID', 'D','PDI','Date']],
    df_2025_ALV[['Sample ID', 'D', 'PDI','Date']],
    df_2025_homogeneity[['Sample ID', 'D', 'PDI','Date']],
    df_2026_05_07[['Sample ID', 'D', 'PDI','Date']],
    ]
)

df_long_term=pd.concat([
df_2022_homogeneity[['Sample ID', 'D','PDI','Date']],
df_2023_24_25[['Sample ID', 'D','PDI','Date']],
    df_2025_ALV[['Sample ID', 'D', 'PDI','Date']],
    df_2025_homogeneity[['Sample ID', 'D', 'PDI','Date']],
    df_2026_05_07[['Sample ID', 'D', 'PDI','Date']],
    ]
)


# Start date is the date of the first measurement
date_start=pd.Timestamp(df_long_term['Date'].min())
df_long_term['time']=(pd.to_datetime(df_long_term['Date']) - date_start).dt.days
df_long_term.head()

In [ ]:
df_2022_homogeneity

In [ ]:
df=df_long_term.copy()
df.rename(columns={'time':'Time'},inplace=True)
df['Time']=df['Time']/30.
#display(df)

# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('Measurement data of hydrodynamic diameter and PDI of the long-therm stability study.')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size


#for col in df.select_dtypes(include='float'):
for col in df[['Sample ID', 'D',  'Date', 'Time']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.1f}")

for col in df[['PDI']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.3f}")

for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        run=paragraph.add_run(str(value))
        run.font.name = 'BAM Klavika Light'#'Arial'
        run.font.size = Pt(10)  # Optional: set font size


save_file=True
if save_file:
    file_name='long_term_data.docx'
    file_path=resultspath / file_name
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df.head()

The stability was assessed according to ISO Guide 35, using a linear regression 
\begin{equation}
y_{lts} = b_0 + b_1 t,
\end{equation}
where $b_0$ is the intercept and $b_1$ the slope. Their estimated standard errors are $s(b_0)$ and $s(b_1)$, respectively.

Next, a test for statistically significant slopes different from zero was conducted.
The $t$-test statistics for slopes $b_1$ significance were calculated using 
\begin{equation}
t_b= \frac{\left| b_1 \right|}{s(b_1)},
\end{equation}
and compared with the critical values at the 95% level of confidence.
Results of linear regression are presented in Figure 5 and values of $t$-statistics in Table xxx. 
For all $D_h$, the slope is not significantly different from zero, and thus, no significant instability is detected.
By contrast, for PDI and zeta potential, the slopes are significantly different from zero, and thus, a significant instability is detected.


In [ ]:
from scipy.stats import linregress

def f_long_term_model_fit_simple(df=None, par=None, save_results=False, verbose=True):
    """ Reference material long-term stability model fit according to ISO GUIDE 35 B.3.2 """

    #x      = df['Time']/365. # time in years
    x      = df['time']/30. # time in month
    x_mean = x.mean()
    sx2    = np.sum((x-x_mean)**2)

    y       = df[par]
    y_mean  = y.mean()
    y_std   = df.describe()[par]['std']
    
    print(f'\nParameter under investitation is {par}')
    n       = len(y)
    print(f'number of data points n = {n}')
    print(f'mean of  {par}  = {y_mean:.2e} ± {y_std:.2e}' )

    # cacluation of slope b1 according to ISO GUIDE 35:2017(E), 
    # slope according to equation (B.14) at page 82 
    b1 = np.sum( (x-x_mean)*(y-y_mean))/sx2
    # intercept according to equation (B.15)
    b0 = y_mean-b1*x_mean
    
    # standard deviations of b0 and b1
    # eq. (B.17) page 82
    s2 = np.sum( (y-b0-b1*x)**2/(n-2))
    s = np.sqrt(s2)
    # s(b1) according to eq. (B.16)
    s_b1= s /np.sqrt(sx2)
    # standard deviation s(b0)
    s_b0= s_b1*np.sqrt((np.sum(x**2))/n)
    
    print('\nLinear regression according to clause B.3 of ISO GUIDE 35:2017(E)')
    print(f'b0 = {b0:.2e} ± {s_b0:.2e} (intercept)')
    print(f'b1 = {b1:.2e} ± {s_b1:.2e} (slope)\n')

    # B.3.3 Inspection and check of assumptions
    # calculation of residuals
    df_fit=pd.DataFrame()
    #n_years = 5 # number of additional years as x-axis
    #df_fit['x_fit'] = np.linspace(x.min(), x.max()+n_years,1000)
    # total number of years on the x-axis
    
    n_time = 60.# maximum time
    x_min = 0 # 20./365. # for logscale
    df_fit['x_fit'] = np.linspace(x_min, n_time,1000)
    df_fit['y_fit'] = b0+b1*df_fit['x_fit']

    # B.3.4 Testing for statistically significant change
    # equation (B.19) at page 82
    t_b1= np.abs(b1)/s_b1
    print('\nTesting for statistical significant change (ISO GUIDE 35: 2017(E) B.3.4)')
    print('t-test statistics for slope significantly different from zero')
    print(f't_b1 = {t_b1:.2e}')

    # ... and comparing this with the two-tailed critical value of Student’s t for n-2 degrees of freedom at the 95 % level 
    # of confidence. If the calculated test statistic tb1 exceeds the critical value, 
    # the slope is considered to be significantly different from zero at the 95 % level of confidence.
    
    # Searching the student-t distribution table for values using Python
    # https://stackoverflow.com/questions/66872064/searching-the-student-t-distribution-table-for-values-using-python
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html
    from scipy.stats import t

    alpha = 0.05  # significance level = 95% 

    degrees_of_freedom = n-2  # degrees of freedom
    t95 = t.ppf(1 - alpha/2, degrees_of_freedom)
    print(f't95 = {t95:.2e} (degrees of freedom = {degrees_of_freedom}, level of significance = 95%)')
    # Consistency check from NIST reference data table on critical values of the Student's t distribution
    # https://www.itl.nist.gov/div898/handbook/eda/section3/eda3672.htm
    # For a two-sided test, we compute 1 - α/2, or 1 - 0.05/2 = 0.975 when α = 0.05
    # for degrees of freedom = 101-2=99 and a significance level of 95%  provides
    # t95 = 1.984

    significance = (t_b1 > t95)
    print('significance: ', significance)
    if significance: 
        print(f't_b1 ({t_b1:.2e}) > t95 ({t95:.2e}) (Interpretation: slope is different from 0)\n')

        # Confidence interval for the regression line (B.3.5 at page 83 of ISO GUIDE 35:2017(E))
        y_fit_delta = t95*s*np.sqrt(1./n + ((df_fit['x_fit']-x.mean())**2/sx2))

        df_fit['y_fit_min']   = df_fit['y_fit'] - y_fit_delta
        df_fit['y_fit_max']   = df_fit['y_fit'] + y_fit_delta
    else:
        print(f't_b1 {t_b1:.2e} < t95 {t95:.2e} (slope is 0)')
        
        # Confidence interval for the regression line (B.3.5 at page 83 of ISO GUIDE 35:2017(E))
        y_fit_delta = t95*s*np.sqrt(1./n + ((df_fit['x_fit']-x.mean())**2/sx2))

        df_fit['y_fit_min']   = df_fit['y_fit'] - y_fit_delta
        df_fit['y_fit_max']   = df_fit['y_fit'] + y_fit_delta

    if verbose:
        # fit linear model
        # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html

        #p-value: float
        #Two-sided p-value for a hypothesis test whose null hypothesis is
        #that the slope is zero, using the Wald Test with a t-distribution of
        #the test statistic.
        model=linregress(x, y)
        #print('linregress(x, y)',linregress(x, y))

        print('\nCalcuation with scipy.stats.linregress')
        print(f'slope    = {model.slope:.2e} ± {model.stderr:.2e}' )
        print(f'intercept= {model.intercept:.2e} ± {model.intercept_stderr:.2e}')
        print(f'rvalue   = {model.rvalue:.2e}')
        print(f'pvalue   = {model.pvalue:.2e}, (Test hypothesis that slope is zero)')

        def f_linear(x, slope, intercept):
            return slope*np.array(x)+intercept

        # Calculate 95% confidence interval on slope and intercept:
        # Two-sided inverse Student's t-distribution
        # p - probability, df - degrees of freedom

        from scipy.stats import t
        tinv = lambda p, df: abs(t.ppf(p/2, df))
        ts = tinv(0.05, len(x)-2)
        print(f"slope     (95%): {model.slope:.2e} ± {ts*model.stderr:.2e}")
        print(f"intercept (95%): {model.intercept:.2e}"f" ± {ts*model.stderr:.2e}")

    # --- plot ---
    fig, ax = plt.subplots(1,1, figsize=[6.4, 4.8])
    
    # Plot with error bars if uncertainties are given.
    if 'u2'+par in df.keys():
        uy=df['u'+par].values
        ax.errorbar(x, y, uy, **BAMMarkerStyles.black, label='data')
    else:
        ax.errorbar(x, y, **BAMMarkerStyles.black, label='data')
    
    
    ax.errorbar(df_fit['x_fit'], y_mean*np.ones(len(df_fit)), color=BAMColors.red, ls=':',lw=2) 

    # --- store the parameters in a Pandas data frame
    df_pars=pd.DataFrame({
        'par' : [par],
        'y_m' : [y_mean],
        's'   : [s],
        'b0'  : [b0],
        's_b0': [s_b0],
        'b1'  : [b1], 
        's_b1': [s_b1],
        't_b1': [t_b1],
        't95' : [t95],
    })
    display(df_pars)

    # -------------------------------------    

    if verbose==True:
        ax.errorbar(x, f_linear(x,model.slope,model.intercept), color=BAMColors.green, label='scipy')
    if 1>0: # plot lines in any case #significance: # plot if slope is significantly larger than 0
        ax.errorbar('x_fit', 'y_fit', data=df_fit, lw=2, color=BAMColors.blue, label='regression')
        #ax.legend()

    plot_verbose=True
    if plot_verbose:
        # one sigma
        x_min = x.min(); 
        x_max = x.max()+1#+365.
        x = np.linspace(x_min, x_max,50)
        y_min = np.ones(len(x))* (y_mean-y_std)
        y_max = np.ones(len(x))* (y_mean+y_std)
        #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.red)
        #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.red)

        # two sigma
        y_min = np.ones(len(x))* (y_mean-2*y_std)
        y_max = np.ones(len(x))* (y_mean+2*y_std)
        #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.blue)
        #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.blue)

        # mean value +- sigma
        #x = [x_max-800/365]
        x = [n_time-1]
        #ax.errorbar(x, y_mean, y_std, **marker_style_red)#, label=r'$\left<D\right>\pm 1\sigma$')
        # mean value +- 2 sigma
        #x = [x_max-200/365]
        x = [n_time]
        #ax.errorbar(x, y_mean, 2*y_std, **marker_style_blue)#, label=r'$\left<D\right>\pm 2\sigma$')

    ax.set_xlabel('Time (months)')
    #ax.set_xscale('log')
    #ax.legend()

    # parameter-dependent axis labelling
    uy = y.max()

    if par=='D':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'$D_h$ (nm)', fontsize=15)
        #ax.set(ylim=(100,250),)

    if par=='PDI':
        #list_xticks=list(df['ID'].values) + [r'$\left<\{}\right>\pm 1\sigma$'.format(par),r'$\left<\{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'{}'.format(par), fontsize=15)
        #ax.set(ylim=(0,.3),)

    if par=='zeta':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        ax.set_ylabel('Zeta potential (mV)', fontsize=15)
        #ax.set(ylim=(-60,-20),)

    save_results=True
    if save_results:
        store_results(resultspath, f"long_term_{par}",
                      (df, df_pars), ("Data", "pars"), saveplot=True)
    plt.show()

    return df_pars

for par in ['D', 'PDI']:
    f_long_term_model_fit_simple(df=df_long_term, par=par, verbose=True, )

### Modification of the Long-Term Stability Study
Following the recommendations provided by Kristin Vogel during the web consultation held on May 6, 2026, the long-term stability study has been revised as outlined below:

The dataset obtained in 2022 has been excluded from the analysis. These measurements were conducted using a Malvern Zetasizer and are therefore not directly comparable with data acquired using the analytical methods implemented in subsequent study phases.

Next demand from Kristin Vogel:
- Calculate mean values for samples measured more than once on the same date

In [ ]:
def f_mean_D_PDI_by_sample_and_date(df):
    """
    If a Sample ID occurs multiple times on the same Date,
    a single entry is created.
    For D and PDI, the mean value is calculated in each case.
    """

    df_mean = (
        df
        .groupby(["Sample ID", "Date"], as_index=False)
        .agg({
            "D": "mean",
            "PDI": "mean",
            "time": "first"
        })
    )

    return df_mean

df_long_term_mean=f_mean_D_PDI_by_sample_and_date(df_long_term)

In [ ]:
from scipy.stats import linregress

def f_long_term_model_fit_simple(df=None, par=None, save_results=False, verbose=True):
    """ Reference material long-term stability model fit according to ISO GUIDE 35 B.3.2 """

    times=sorted(df_long_term['time'].unique())
    times=[int(t) for t in times]
    print(f'samples were measured at {times} months')
    
    # remove the values from January 2022
    # time at start of long-term study
    t0=569. 
    df1=df.copy()
    df1=df1[df1['time']<t0]
    df1['time']=df1['time']-t0

    df2=df.copy()
    df2=df2[df2['time']>=t0]
    df=df2
    df['time']=df['time']-t0
    # --------------------------------------------------
    
    #x      = df['Time']/365. # time in years
    x      = df['time']/30. # time in month
    x_mean = x.mean()
    sx2    = np.sum((x-x_mean)**2)

    y       = df[par]
    y_mean  = y.mean()
    y_std   = df.describe()[par]['std']
    
    print(f'\nParameter under investitation is {par}')
    n       = len(y)
    print(f'number of data points n = {n}')
    print(f'mean of  {par}  = {y_mean:.2e} ± {y_std:.2e}' )

    # cacluation of slope b1 according to ISO GUIDE 35:2017(E), 
    # slope according to equation (B.14) at page 82 
    b1 = np.sum( (x-x_mean)*(y-y_mean))/sx2
    # intercept according to equation (B.15)
    b0 = y_mean-b1*x_mean
    
    # standard deviations of b0 and b1
    # eq. (B.17) page 82
    s2 = np.sum( (y-b0-b1*x)**2/(n-2))
    s = np.sqrt(s2)
    # s(b1) according to eq. (B.16)
    s_b1= s /np.sqrt(sx2)
    # standard deviation s(b0)
    s_b0= s_b1*np.sqrt((np.sum(x**2))/n)
    
    print('\nLinear regression according to clause B.3 of ISO GUIDE 35:2017(E)')
    print(f'b0 = {b0:.2e} ± {s_b0:.2e} (intercept)')
    print(f'b1 = {b1:.2e} ± {s_b1:.2e} (slope)\n')

    # B.3.3 Inspection and check of assumptions
    # calculation of residuals
    df_fit=pd.DataFrame()
    #n_years = 5 # number of additional years as x-axis
    #df_fit['x_fit'] = np.linspace(x.min(), x.max()+n_years,1000)
    # total number of years on the x-axis
    
    n_time = 60.# maximum time
    x_min =  0 # 20./365. # for logscale
    df_fit['x_fit'] = np.linspace(x_min, n_time,1000)
    df_fit['y_fit'] = b0+b1*df_fit['x_fit']

    # B.3.4 Testing for statistically significant change
    # equation (B.19) at page 82
    t_b1= np.abs(b1)/s_b1
    print('\nTesting for statistical significant change (ISO GUIDE 35: 2017(E) B.3.4)')
    print('t-test statistics for slope significantly different from zero')
    print(f't_b1 = {t_b1:.2e}')

    # ... and comparing this with the two-tailed critical value of Student’s t for n-2 degrees of freedom at the 95 % level 
    # of confidence. If the calculated test statistic tb1 exceeds the critical value, 
    # the slope is considered to be significantly different from zero at the 95 % level of confidence.
    
    # Searching the student-t distribution table for values using Python
    # https://stackoverflow.com/questions/66872064/searching-the-student-t-distribution-table-for-values-using-python
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html
    from scipy.stats import t

    alpha = 0.05  # significance level = 95% 

    degrees_of_freedom = n-2  # degrees of freedom
    t95 = t.ppf(1 - alpha/2, degrees_of_freedom)
    print(f't95 = {t95:.2e} (degrees of freedom = {degrees_of_freedom}, level of significance = 95%)')
    # Consistency check from NIST reference data table on critical values of the Student's t distribution
    # https://www.itl.nist.gov/div898/handbook/eda/section3/eda3672.htm
    # For a two-sided test, we compute 1 - α/2, or 1 - 0.05/2 = 0.975 when α = 0.05
    # for degrees of freedom = 101-2=99 and a significance level of 95%  provides
    # t95 = 1.984

    significance = (t_b1 > t95)
    print('significance: ', significance)
    if significance: 
        print(f't_b1 ({t_b1:.2e}) > t95 ({t95:.2e}) (Interpretation: slope is different from 0)\n')

        # Confidence interval for the regression line (B.3.5 at page 83 of ISO GUIDE 35:2017(E))
        y_fit_delta = t95*s*np.sqrt(1./n + ((df_fit['x_fit']-x.mean())**2/sx2))

        df_fit['y_fit_min']   = df_fit['y_fit'] - y_fit_delta
        df_fit['y_fit_max']   = df_fit['y_fit'] + y_fit_delta
    else:
        print(f't_b1 {t_b1:.2e} < t95 {t95:.2e} (slope is 0)')
        
        # Confidence interval for the regression line (B.3.5 at page 83 of ISO GUIDE 35:2017(E))
        y_fit_delta = t95*s*np.sqrt(1./n + ((df_fit['x_fit']-x.mean())**2/sx2))

        df_fit['y_fit_min']   = df_fit['y_fit'] - y_fit_delta
        df_fit['y_fit_max']   = df_fit['y_fit'] + y_fit_delta

    if verbose:
        # fit linear model
        # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html

        #p-value: float
        #Two-sided p-value for a hypothesis test whose null hypothesis is
        #that the slope is zero, using the Wald Test with a t-distribution of
        #the test statistic.
        model=linregress(x, y)
        #print('linregress(x, y)',linregress(x, y))

        print('\nCalcuation with scipy.stats.linregress')
        print(f'slope    = {model.slope:.2e} ± {model.stderr:.2e}' )
        print(f'intercept= {model.intercept:.2e} ± {model.intercept_stderr:.2e}')
        print(f'rvalue   = {model.rvalue:.2e}')
        print(f'pvalue   = {model.pvalue:.2e}, (Test hypothesis that slope is zero)')

        def f_linear(x, slope, intercept):
            return slope*np.array(x)+intercept

        # Calculate 95% confidence interval on slope and intercept:
        # Two-sided inverse Student's t-distribution
        # p - probability, df - degrees of freedom

        from scipy.stats import t
        tinv = lambda p, df: abs(t.ppf(p/2, df))
        ts = tinv(0.05, len(x)-2)
        print(f"slope     (95%): {model.slope:.2e} ± {ts*model.stderr:.2e}")
        print(f"intercept (95%): {model.intercept:.2e}"f" ± {ts*model.stderr:.2e}")

    # --- plot ---
    fig, ax = plt.subplots(1,1, figsize=[6.4, 4.8])
    
    # Plot with error bars if uncertainties are given.
    if 'u2'+par in df.keys():
        uy=df['u'+par].values
        ax.errorbar(x, y, uy, **BAMMarkerStyles.black, label='data')
    else:
        ax.errorbar(x, y, **BAMMarkerStyles.black, label='data')

        x1 = df1['time']/30.
        y1  = df1[par]
        markerstyle={'color': '#002832', 'linestyle': '', 'marker': '^', 'fillstyle': 'none', 'markersize': 5, 
                     'markeredgewidth': 1, 'markeredgecolor': '#002832'}
        ax.errorbar(x1, y1, **markerstyle)
    
    ax.errorbar(df_fit['x_fit'], y_mean*np.ones(len(df_fit)), color=BAMColors.red, ls=':',lw=2) 

    # --- store the parameters in a Pandas data frame
    df_pars=pd.DataFrame({
        'par' : [par],
        'y_m' : [y_mean],
        's'   : [s],
        'b0'  : [b0],
        's_b0': [s_b0],
        'b1'  : [b1], 
        's_b1': [s_b1],
        't_b1': [t_b1],
        't95' : [t95],
    })
    display(df_pars)

    # -------------------------------------    

    if verbose==True:
        ax.errorbar(x, f_linear(x,model.slope,model.intercept), color=BAMColors.green, label='scipy')
    if 1>0: # plot lines in any case #significance: # plot if slope is significantly larger than 0
        ax.errorbar('x_fit', 'y_fit', data=df_fit, lw=2, color=BAMColors.blue, label='regression')
        #ax.legend()

    plot_verbose=True
    if plot_verbose:
        # one sigma
        x_min = x.min(); 
        x_max = x.max()+1#+365.
        x = np.linspace(x_min, x_max,50)
        y_min = np.ones(len(x))* (y_mean-y_std)
        y_max = np.ones(len(x))* (y_mean+y_std)
        #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.red)
        #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.red)

        # two sigma
        y_min = np.ones(len(x))* (y_mean-2*y_std)
        y_max = np.ones(len(x))* (y_mean+2*y_std)
        #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.blue)
        #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.blue)

        # mean value +- sigma
        #x = [x_max-800/365]
        x = [n_time-1]
        #ax.errorbar(x, y_mean, y_std, **marker_style_red)#, label=r'$\left<D\right>\pm 1\sigma$')
        # mean value +- 2 sigma
        #x = [x_max-200/365]
        x = [n_time]
        #ax.errorbar(x, y_mean, 2*y_std, **marker_style_blue)#, label=r'$\left<D\right>\pm 2\sigma$')

    ax.set_xlabel('Time (months)')
    #ax.set_xscale('log')
    #ax.legend()

    # parameter-dependent axis labeling
    uy = y.max()

    if par=='D':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'$D_h$ (nm)', fontsize=15)
        #ax.set(ylim=(100,250),)

    if par=='PDI':
        #list_xticks=list(df['ID'].values) + [r'$\left<\{}\right>\pm 1\sigma$'.format(par),r'$\left<\{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'{}'.format(par), fontsize=15)
        #ax.set(ylim=(0,.3),)

    if par=='zeta':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        ax.set_ylabel('Zeta potential (mV)', fontsize=15)
        #ax.set(ylim=(-60,-20),)

    save_results=True
    if save_results:
        store_results(resultspath, f"long_term_{par}",
                      (df, df_pars), ("Data", "pars"), saveplot=True)
    plt.show()

    return df_pars

for par in ['D', 'PDI']:
    f_long_term_model_fit_simple(df=df_long_term_mean, par=par, verbose=True, )

In [ ]:
df_lts_D=pd.read_excel(resultspath / 'long_term_D_pars.xlsx', index_col=0)
display(df_lts_D)

df_lts_PDI=pd.read_excel(resultspath / 'long_term_PDI_pars.xlsx', index_col=0)
display(df_lts_PDI)

#df_lts_zeta=pd.read_excel(resultspath / 'long_term_stability' / 'long_term_zeta_pars.xlsx', index_col=0)
#display(df_lts_zeta)

# time of stability study
t_m=12.
# time of validity
t_cert=12.

l_dfs=[]
for item in ['D', 'PDI',]:
    df=pd.read_excel(resultspath / ('long_term_'+item+'_pars.xlsx'), index_col=0)
    l_dfs.append(df)
df_pars_lts=pd.concat(l_dfs)
df_pars_lts.reset_index(inplace=True, drop=True)
df_pars_lts['u_lts']= df_pars_lts['s_b1']*(t_m+t_cert)
display(df_pars_lts)

In [ ]:
## Save as Word files 
#- subsection 7.3
df_long_term_mean

df_long_term_mean = (
    df_long_term_mean
    .sort_values(by="time")
    .reset_index(drop=True)
)

start_month=19. # first month of measurement used
df_long_term_mean["time_30"] = (df_long_term_mean["time"] / 30).round(1)-start_month
df_long_term_mean["D"]=df_long_term_mean["D"].round(1)
df_long_term_mean["PDI"]=df_long_term_mean["PDI"].round(3)

# data utilized 
df_utilized=df_long_term_mean[(df_long_term_mean['time_30']>=0)][['Sample ID', 'D', 'PDI', 'Date', 'time_30']]

# data not utilized
df_not_utilized=df_long_term_mean[(df_long_term_mean['time_30']<0.)][['Sample ID', 'D', 'PDI', 'Date', 'time_30']]

In [ ]:
from pathlib import Path

from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


def f_set_run_font(run, font_name="BAM Klavika Light", font_size=None, bold=None):
    """
    Set font name, size and bold formatting for a Word run.
    """

    run.font.name = font_name

    # Helps Word apply the font reliably
    run._element.rPr.rFonts.set(qn("w:ascii"), font_name)
    run._element.rPr.rFonts.set(qn("w:hAnsi"), font_name)
    run._element.rPr.rFonts.set(qn("w:cs"), font_name)

    if font_size is not None:
        run.font.size = Pt(font_size)

    if bold is not None:
        run.bold = bold


def f_set_cell_border(cell, **kwargs):
    """
    Set borders for a table cell.
    """

    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()

    tcBorders = tcPr.first_child_found_in("w:tcBorders")

    if tcBorders is None:
        tcBorders = OxmlElement("w:tcBorders")
        tcPr.append(tcBorders)

    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        if edge in kwargs:
            edge_data = kwargs[edge]

            tag = "w:{}".format(edge)
            element = tcBorders.find(qn(tag))

            if element is None:
                element = OxmlElement(tag)
                tcBorders.append(element)

            for key, value in edge_data.items():
                element.set(qn("w:{}".format(key)), str(value))


def f_remove_cell_borders(cell):
    """
    Remove all borders from a table cell.
    """

    f_set_cell_border(
        cell,
        top={"val": "nil"},
        left={"val": "nil"},
        bottom={"val": "nil"},
        right={"val": "nil"},
        insideH={"val": "nil"},
        insideV={"val": "nil"},
    )


def f_format_value(value, column_name):
    """
    Format values depending on the column.
    """

    if column_name == "D (nm)":
        return f"{float(value):.1f}"

    elif column_name == "PDI":
        return f"{float(value):.3f}"

    elif column_name == "Time (month)":
        return f"{float(value):.1f}"

    else:
        return str(value)


def f_save_dataframe_as_word(df, file_name="df_utilized.docx"):
    """
    Save dataframe as Word table without grid lines.
    Only a horizontal line below the header row is shown.
    Font: BAM Klavika Light
    """

    font_name = "BAM Klavika Light"

    # Copy dataframe so original dataframe is not changed
    df_word = df.copy()

    # Rename columns
    df_word = df_word.rename(
        columns={
            "D": "D (nm)",
            "time_30": "Time (month)",
        }
    )

    # Optional: sort by Time (month)
    if "Time (month)" in df_word.columns:
        df_word = (
            df_word
            .sort_values(by="Time (month)")
            .reset_index(drop=True)
        )

    # Create output folder if needed
    file_name = Path(file_name)
    file_name.parent.mkdir(parents=True, exist_ok=True)

    # Create Word document
    doc = Document()

    # Set default Normal style font
    normal_style = doc.styles["Normal"]
    normal_style.font.name = font_name
    normal_style._element.rPr.rFonts.set(qn("w:ascii"), font_name)
    normal_style._element.rPr.rFonts.set(qn("w:hAnsi"), font_name)
    normal_style._element.rPr.rFonts.set(qn("w:cs"), font_name)
    normal_style.font.size = Pt(9)

    # Add title
    heading = doc.add_heading(level=1)
    heading.alignment = WD_ALIGN_PARAGRAPH.LEFT

    heading_run = heading.add_run("")
    f_set_run_font(
        heading_run,
        font_name=font_name,
        font_size=16,
        bold=False
    )

    # Add table
    table = doc.add_table(
        rows=1,
        cols=len(df_word.columns)
    )

    table.alignment = WD_TABLE_ALIGNMENT.CENTER

    # Header row
    header_cells = table.rows[0].cells

    for i, column_name in enumerate(df_word.columns):
        cell = header_cells[i]
        cell.text = ""
        cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

        # Remove all borders first
        f_remove_cell_borders(cell)

        # Add only horizontal line below header
        f_set_cell_border(
            cell,
            bottom={
                "val": "single",
                "sz": "8",
                "space": "0",
                "color": "000000",
            },
        )

        paragraph = cell.paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

        run = paragraph.add_run(str(column_name))
        f_set_run_font(
            run,
            font_name=font_name,
            font_size=10,
            bold=True
        )

    # Data rows
    for _, row in df_word.iterrows():
        cells = table.add_row().cells

        for i, value in enumerate(row):
            column_name = df_word.columns[i]
            text = f_format_value(value, column_name)

            cell = cells[i]
            cell.text = ""
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

            # Remove all borders from body cells
            f_remove_cell_borders(cell)

            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

            run = paragraph.add_run(text)
            f_set_run_font(
                run,
                font_name=font_name,
                font_size=9,
                bold=False
            )

    # Save Word file
    doc.save(file_name)

    print(f"Saved Word file: {file_name}")


f_save_dataframe_as_word(
    df_utilized,
    file_name=resultspath / "df_long_term_utilized.docx"
)

f_save_dataframe_as_word(
    df_not_utilized,
    file_name=resultspath / "df_long_term_not_utilized.docx"
)

### Estimation of u_lts
 Uncertainty of $u_{lts}$
 
 \begin{equation}
 u_{lts} = \sqrt{u_{lts,1}^2 + u_{lts,2}^2} 
 \end{equation}
 with (see eq. (10) at page 44 of ISO Guide 35)
 \begin{equation}
 u_{lts,1} = s(b_1) (t_{m1}+t_{cert}) 
 \end{equation}
 and (from comment by Kristin Vogel on 2025-09-17, i.e., under the assumption of equal distribution of the values (rectangular function))
 \begin{equation}
 u_{lts,2} = \frac{b_1}{2 \sqrt{3}} t_{cert} 
 \end{equation}
 In the absence of a significant trend is $u_{lts,2}=0$
 
- $t_{m1}$ is the time interval between value assignment and the initial stability monitoring point (1 years)
- $t_{cert}$ is the planned lifetime (here it is 2 years)

In [ ]:
b1=df_pars_lts[df_pars_lts['par']=='D']['b1'].values[0]
s_b1=df_pars_lts[df_pars_lts['par']=='D']['s_b1'].values[0]
print(f'b1 = {b1:2.3f}, s_b1 = {s_b1:2.3f}')

# without data from 2022 as recommended by Vogel
# with data from 2026-05-07

t_m = 12.
t_cert=24.

u_lts1= s_b1 *(t_m + t_cert)
u_lts2=b1/(2*np.sqrt(3))*t_cert
u_lts=np.sqrt(u_lts1**2 + u_lts2**2)                  
print(f'u_lts1 = {u_lts1:2.3f}, u_lts2 = {u_lts2:2.3f}, u_lts = {u_lts:2.3f}')

# Calculation of combined uncertainty

The combined uncertainty $u_c$ was calculated according to Equation (xxx), using the numerical values summarized in Table xxx. This equation is a combination of the standard uncertainty due to characterization, the contribution from variation between bottles, and the contribution from long-term stability. Furthermore, the certified value xCRM can be assigned as ychar since no between-unit variation or stability effects need to be regarded (Equation xxx).

\begin{equation}
u_c^2=  u_{\textrm{char}}^2 + u_{\textrm{hom}}^2 + u_{\textrm{lts}}^2
\end{equation}
where the $u_i$ are the different uncertainty contributions.
The expanded uncertainty is 
\begin{equation}
U = k \times u_c
\end{equation}
with an expansion factor of $k=2$. 

Note: At the consultation with Kristin Vogel (2026-05-06), she advised taking a 7% uncertainty in $D_{RM} $ for $u_c$.


In [ ]:
print(f'D_RM = {y_hom:2.0f} nm') # from homogeneity study in September 2025

print(f'value from three instruments: u_char = {u_char:2.2f} nm')

u_char=0.07*y_hom
print(f'value as 7% of D_RM: u_char = {u_char:2.2f} nm')
print('plausibility check')


print(f'u_hom = {u_hom} nm')
print(f'u_lts = {u_lts:2.2f} nm')

u_c=np.sqrt(np.sum([u_hom**2,u_char**2, u_lts**2]))
U = 2*u_c

print(f'u_c = {u_c:2.0f} nm' )
print(f'U = {U:2.0f} nm' )

In [ ]:
D_Litesizer_500=df_2025_homogeneity['D'][0:10].describe()['mean']
D_ALV=df_2025_ALV['D'].describe()['mean']
D_ZetaSizer=df_2022_homogeneity['D'][0:31].describe()['mean']

l_Dchar=[D_Litesizer_500, D_ALV, D_ZetaSizer]
D_mean=np.mean(l_Dchar)
deviations=[ D_mean-item for item in l_Dchar]

# Heat sterilization
Comment Kristin Vogel  
Die Anova sollte prüfen ob sich die 3 Gruppen „before“, „after heat 1“ „after heat 2“ unterscheiden. In der aktuellen Auswertung wird geprüft, ob die samples unterschiedlich sind (also homogenität). Für eine statistische Aussage zum Einfluss der Erhitzung sollte die Anova mit der Hypothese wiederholt werden, dass die 3 Gruppen (vorher, nachher1, nachher2) zur gleichen Verteilung gehören. (Spalten/Zeilen für die ANOVA drehen). 

Hypothesis: Heat sterilization does not alter hydrodynamic diameter, PDI, or zeta potential

In [ ]:
def f_ANOVA_heat_sterilization(df):
    """Perform one-way ANOVA according to ISO Guide 35
    
    The one-way ANOVA tests the null hypothesis that two or more groups have the same population mean. 
    The sample data for control are from
    ISO GUIDE 35:2017(E) Reference materials — Guidance for characterization and assessment of homogeneity and stability
    Page 89, Table C.1 — Measurement data of a between-unit homogeneity study of chromium in soil
    
    see also
    https://en.wikipedia.org/wiki/One-way_analysis_of_variance
    """
    df = df.copy() # a copy of the dataframe provided

    # Step 1: Calculate the mean within each group;  groups are the bottles:
    cols = list(df.keys())
    a = len(df) # number of groups
    n = len(df.columns)# 3 # where n is the number of data values per group. 
    print('Number of groups a =', a, '\nNumber of data per group n =',n )
    
    # Create a summary table as Excel produces it for "Anova: Single Factor"
    d_sums, d_means = {}, {} # dictionary containing mean of each group
    for i in range(a):
        # sums
        #d_sums['Y'+str(i)] =(df.iloc[i][cols[1]]+df.iloc[i][cols[2]]+df.iloc[i][cols[3]])
        d_sums['Y'+str(i)]=np.sum([ df.iloc[i][cols[j]] for j in range(n)])
        # means
        #d_means['Y'+str(i)]= (df.iloc[i][cols[1]]+df.iloc[i][cols[2]]+df.iloc[i][cols[3]])/n
        d_means['Y'+str(i)]=d_sums['Y'+str(i)]/n

        
    # Step 2: Calculate the overall mean: 
    Y = sum([ d_means[key] for key in d_means.keys()])/len(d_means)
    
    print('Overall mean =',f_round_sig(Y,5))
    
    # Step 3: Calculate the "between-group" sum of squared differences: 
    SS_between = sum([ n*(d_means[key]-Y)**2 for key in d_means.keys()])
    # The between-group degrees of freedom is one less than the number of groups
    df_between = a-1
    MS_between = SS_between/df_between
    #print('Source of variance between-group: SS =',f_round_sig(SS_between,5), ', df =', df_between, ', MS =',f_round_sig(MS_between,5))

    # Step 4: Calculate the "within-group" sum of squares. 
    # sum of the values within each group
    df['Sum'] = d_sums.values()
    # mean value within each group
    df['Mean'] = d_means.values()
    # sum of squares within each group
    df['SS'] = (df[df.keys()[1]]-df['Mean'])**2+(df[df.keys()[2]]-df['Mean'])**2+(df[df.keys()[3]]-df['Mean'])**2
    # standard deviation of the mean within each group
    df['Variance'] = df['SS']/(n-1.)
    df['Variance']=df['Variance'].astype(float)
    df['Sigma'] = np.sqrt(df['Variance'])
    # sum of squares within group
    SS_within = df['SS'].sum()
    # The within-group degrees of freedom is a*(n-1)
    df_within = a*(n-1)
    # Thus the within-group mean square value is 
    MS_within = SS_within/df_within
    #print('Source of variance within-group:  SS =',f_round_sig(SS_within,5), ', df =', df_within, ', MS =',f_round_sig(MS_within,5))
    F_value = MS_between/MS_within
    #print('F_value = ', f_round_sig(F_value,5))

    # find the critical F-Value
    # https://stackoverflow.com/questions/39813470/f-test-with-python-finding-the-critical-value
    import scipy.stats
    # Confidence level of 95% is q=1.-0.05
    F_crit = scipy.stats.f.ppf(q=1-0.05, dfn=df_between, dfd=df_within)
    #print("F_crit  = ", f_round_sig(F_crit,5), "(critical F-Value, corresponding to a confidence level of ",
    #scipy.stats.f.cdf(F_crit, dfn=df_between, dfd=df_within)*100,'%)')

    # check of F value and determination of p-value 
    array4f_oneway = [[df.iloc[i][cols[1]], df.iloc[i][cols[2]],  df.iloc[i][cols[3]]] for i in range(len(df))]
    F_value_2, p_value = f_oneway(*array4f_oneway) 
    #print('p-value =', p_value)
    
    # The between-unit standard deviation (see. formula C.2 at page 90 of ISO GUIDE 35:2017(E))
    if MS_between > MS_within:
        s_bb = np.sqrt((MS_between-MS_within)/n)
    else: s_bb = 0.
    
    # The repeatability standard deviation (see formula C.3 at page 90 of ISO GUIDE 35:2017(E)))
    s_r  = np.sqrt(MS_within)
    
    # Table according to ISO Guide 35:2017(E) Table C.2, page 89
    df_ANOVA_table = pd.DataFrame( 
        {   'Overall Mean':[f_round_sig(Y,5),'',''],
            'Overall Std': [f_round_sig(
                    np.sqrt((SS_between+SS_within)/(df_between+df_within)),3),'',''],
            'Source of variation':['Between bottles','Within bottles','Total'], 
            'Sum of squares':[
                f_round_sig(SS_between,5),
                f_round_sig(SS_within,5),
                f_round_sig((SS_between+SS_within),5)
                ], 
            'Degrees of freedom':[df_between,df_within,df_between+df_within], 
            'Mean square':[ f_round_sig(MS_between,5), f_round_sig(MS_within,5), ''],
            'Standard deviation':[f_round_sig(s_bb,4), f_round_sig(s_r,4),' '],
            'F':[f_round_sig(F_value,5),'',''], 
            'Fcrit':[f_round_sig(F_crit,5),'',''],
            'p-value':[p_value,'',''],
        })
    
    return df, df_ANOVA_table

In [ ]:
d_data1=f_read_DLS(basepath / 'data_2025-06-30', files='[pP]*.xlsx', verbose=False)
df_D1=f_DLS_results(d_data1)
d_data2=f_read_DLS(basepath / 'data_2025-07-07', files='[pP]*.xlsx', verbose=False)
df_D2=f_DLS_results(d_data2)
df_D=pd.concat([df_D1,df_D2])
df_D.rename(columns={'Dh': 'D', 'sample_ID': 'ID'}, inplace=True)

fig, ax = plt.subplots()
ax.errorbar(df_D[0:10].index, df_D['D'][ 0:10], **BAMMarkerStyles.blue , label='before heat treatment')
ax.errorbar(df_D[0:10].index, df_D['D'][10:20], **BAMMarkerStyles.red, label='after heat treatment 1')
ax.errorbar(df_D[0:10].index, df_D['D'][20:30], **BAMMarkerStyles.green, label='after heat treatment 2')
ax.set(
    xlabel='Sample',
    ylabel='$D_h$ (nm)',
    ylim=(100,250)
)
ax.legend(frameon=False)

df_D.head(2)

In [ ]:
d_data1=f_read_DLS(basepath / 'data_2025-06-30', files='[zZ]*.xlsx', verbose=False)
df_zeta1=f_zeta_results(d_data1)
d_data2=f_read_DLS(basepath / 'data_2025-07-07', files='[zZ]*.xlsx', verbose=False)
df_zeta2=f_zeta_results(d_data2)
df_zeta=pd.concat([df_zeta1,df_zeta2])
df_zeta.reset_index(inplace=True, drop=True)
df_zeta.head(2)

In [ ]:
# Data before heat treatment, DLS data obtained with Litesizer 500
df_2025_June_before_heat=df_D[0:10][['ID', 'date of measurement','D','PDI' ]].merge(df_zeta[0:10][['ID', 'date of measurement','zeta' ]]).copy()

# Data after heat treatment, DLS data obtained with Litesizer 500 
df_2025_June_after_heat=df_D[10:20][['ID', 'date of measurement','D','PDI' ]].merge(df_zeta[10:20][['ID', 'date of measurement','zeta' ]]).copy()

# Data after heat treatment, DLS data obtained with the instrument from ALV Langen
df_2025_July_after_heat2=df_D[20:30][['ID', 'date of measurement','D','PDI' ]].merge(df_zeta[20:30][['ID', 'date of measurement','zeta' ]]).copy()

In [ ]:
# heat stability study
def f_summarize(df_input):
    df=pd.DataFrame()
    for item in ['D', 'PDI', 'zeta']:
        df_x=df_input.groupby('date of measurement')[item].agg(['mean', 'std'])
        df[item]=df_x['mean']
        df['u'+item]=df_x['std']
    df.reset_index(inplace=True)
    df['date of measurement']=pd.to_datetime(df['date of measurement']).dt.date
    return df

In [ ]:
# Before heat treatment
df_2025_June_before_heat_summary=f_summarize(df_2025_June_before_heat)
print('Before heat treatment at 121°C for 90 min')
display(df_2025_June_before_heat_summary)

print('After heat treatment at 121°C for 90 min')
df_2025_June_after_heat_summary=f_summarize(df_2025_June_after_heat)
display(df_2025_June_after_heat_summary)

print('Second measurement series after heat treatment at 121°C for 90 min')
df_2025_July_after_heat2_summary=f_summarize(df_2025_July_after_heat2)
display(df_2025_July_after_heat2_summary)

df_heat_summary=pd.concat([df_2025_June_before_heat_summary, df_2025_June_after_heat_summary, df_2025_July_after_heat2_summary ])
df_heat_summary['time']=l_heat=['before sterilization', '3 h after sterilization', '24 h after sterilization']
df_heat_summary=df_heat_summary[['date of measurement','time', 'D', 'uD', 'PDI', 'uPDI', 'zeta', 'uzeta']].reset_index(drop=True)
display(df_heat_summary)

In [ ]:
def f_plot_heat_summary(df, par='D'):
    fig, ax = plt.subplots()
    ax.errorbar(df_heat_summary.index, par, 'u'+par, data=df, **BAMMarkerStyles.black, label='data')
    
    if par=='D':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'$D_h$ (nm)', fontsize=15)
        #ax.set(ylim=(100,250),)

    if par=='PDI':
        #list_xticks=list(df['ID'].values) + [r'$\left<\{}\right>\pm 1\sigma$'.format(par),r'$\left<\{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'{}'.format(par), fontsize=15)
        #ax.set(ylim=(0,.3),)

    if par=='zeta':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        ax.set_ylabel('Zeta potential (mV)', fontsize=15)
        #ax.set(ylim=(-60,-20),)

        save_results=False
        if save_results:
            store_results(resultspath / "long_term_stability", f"long_term_{par}",
                          (df, df2), ("Data", "Table"), saveplot=True)
        plt.show()

f_plot_heat_summary(df_heat_summary,'D')
f_plot_heat_summary(df_heat_summary,'PDI')
f_plot_heat_summary(df_heat_summary,'zeta')

In [ ]:
df = df_heat_summary.copy()

# change font size for axes
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['font.size']= 8

cm = 1 / 2.54  # centimeters to inches
fig, (ax, ax2, ax3) = plt.subplots(1,3, figsize=(15 * cm, 8 * cm))

ax.errorbar(df.index, 'D', 'uD', data=df, **BAMMarkerStyles.black, label='$D_h$')
#list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
ax.set_ylabel(r'$D_h$ (nm)', fontsize=12)
ax.set(xlim=(-.5,2.5),)
ax.set_xticks(ticks=[0, 1, 2], 
              labels=['before\nheat', 'after\nheat 1', 'after\nheat 2',],fontsize=8)

ax2.errorbar(df.index, 'PDI', 'uPDI', data=df, **BAMMarkerStyles.black, label='PDI')
#list_xticks=list(df['ID'].values) + [r'$\left<\{}\right>\pm 1\sigma$'.format(par),r'$\left<\{}\right>\pm 2\sigma$'.format(par)]
ax2.set_ylabel(r'PDI', fontsize=10)
ax2.set(xlim=(-.5,2.5),)
ax2.set_xticks(ticks=[0, 1, 2], 
              labels=['before\nheat', 'after\nheat 1', 'after\nheat 2',],fontsize=8)

ax3.errorbar(df.index, 'zeta', 'uzeta', data=df, **BAMMarkerStyles.black, label='zeta potential')
#list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
ax3.set_ylabel('Zeta potential (mV)', fontsize=10)
ax3.set(xlim=(-.5,2.5),)
ax3.set_xticks(ticks=[0, 1, 2], 
              labels=['before\nheat', 'after\nheat 1', 'after\nheat 2',], fontsize=8)
plt.tight_layout()

save_fig=True
if save_fig:
    fig.savefig(resultspath / "heat_stability.png", dpi=600)


### Export as word table

In [ ]:
df=df_heat_summary.copy()
display(df)
# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('Data of hydrodynamic diameter, PDI, and zeta potential from the figure above.')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size

#for col in df.select_dtypes(include='float'):

#for col in df[['before heat','after heat 1', 'after heat 2','Sum','Mean','SS']].select_dtypes(include='float'):
#    df[col] = df[col].map(lambda x: f"{x:03.1f}")

df['D'] = df['D'].map(lambda x: f"{x:02.0f}")
df['uD'] = df['uD'].map(lambda x: f"{x:02.0f}")

df['PDI']=df['PDI'].astype(float)
df['PDI'] = df['PDI'].map(lambda x: f"{x:02.2f}")
df['uPDI'] = df['uPDI'].map(lambda x: f"{x:02.2f}")

df['zeta'] = df['zeta'].map(lambda x: f"{x:02.0f}")
df['uzeta'] = df['uzeta'].map(lambda x: f"{x:01.0f}")


for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        run=paragraph.add_run(str(value))
        run.font.name = 'BAM Klavika Light'#'Arial'
        run.font.size = Pt(10)  # Optional: set font size


save_file=True
if save_file:
    file_name='heat_stability_table'+'.docx'
    file_path=resultspath / file_name
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df

In [ ]:
d_ANOVA_heat={}
for par in ['D', 'PDI', 'zeta']:
    df_res1=df_2025_June_before_heat[['ID', par]].rename(columns={par: 'before heat'})
    df_res2=df_2025_June_after_heat[['ID', par]].rename(columns={par: 'after heat 1'})
    df_res3=df_2025_July_after_heat2[['ID', par]].rename(columns={par: 'after heat 2'})
    df_merged=df_res1.merge(df_res2).merge(df_res3)
    d_ANOVA_heat[par]=df_merged.copy()
d_ANOVA_heat.keys()

In [ ]:
df_ANOVA_D=d_ANOVA_heat['D'].T.loc[:][1:]
#display(df_ANOVA_D)

df, df_ANOVA_table = f_ANOVA_heat_sterilization(df_ANOVA_D)
display(df)
df_ANOVA_table

In [ ]:
df_ANOVA_D=d_ANOVA_heat['PDI'].T.loc[:][1:]
#display(df_ANOVA_D)

df, df_ANOVA_table = f_ANOVA_heat_sterilization(df_ANOVA_D)
display(df)
df_ANOVA_table

In [ ]:
df_ANOVA_D=d_ANOVA_heat['zeta'].T.loc[:][1:]
#display(df_ANOVA_D)

df, df_ANOVA_table = f_ANOVA_heat_sterilization(df_ANOVA_D)
display(df)
df_ANOVA_table